# 03 — Exploratory analysis

**Question:** what does the movement signal actually look like for each behaviour, and is there visible separation before any model is fitted?

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)
from smartnet import config
from smartnet.data import loader
from smartnet.visualization import plots as P

df = loader.load_analysis_frame()
events = loader.event_summary(df, 'motion_5cat')

In [ ]:
fig = P.plot_class_balance(df, 'motion_5cat'); plt.show()

In [ ]:
fig = P.plot_event_duration(events); plt.show()

Events are short. Most are a handful of seconds, and adjacent epochs within one event share ±10 seconds of context — so they are near-duplicates of each other. Notebook 05 quantifies what that does to a random train/test split.

In [ ]:
fig = P.plot_temporal_coverage(df, 'motion_5cat'); plt.show()

Motions were staged during daytime sessions; *Nothing* was sampled across all 24 hours. This is a property of the data-collection protocol, and it means hour-of-day would be a leaky feature — it is deliberately excluded from the model.

In [ ]:
fig = P.plot_signal_examples(df, 'motion_5cat'); plt.show()

## Feature distributions by class

In [ ]:
names = config.LABEL_MAPS['motion_5cat']
key = ['sum_vectormagnitudes','stdvz','meanzover10vector_back','sumover10vector_forward']
summary = df.groupby(df.motion_5cat.map(names))[key].agg(['mean','std']).round(3)
summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, col in zip(axes.ravel(), key):
    for i, (name, sub) in enumerate(df.groupby(df.motion_5cat.map(names))):
        ax.hist(sub[col], bins=30, alpha=0.55, label=name, color=P.PALETTE[i % len(P.PALETTE)])
    ax.set_title(col, fontsize=9); ax.set_yscale('log')
axes[0,0].legend(fontsize=7, frameon=False)
plt.tight_layout(); plt.show()

*Nothing* separates cleanly on every feature — it is near-zero movement. The four motion classes overlap heavily with one another, which previews the result: detecting *that* something happened is easy; identifying *which* motion is not.

## Correlation structure

In [ ]:
corr = df[config.WINDOW_AGGREGATE_FEATURES + config.CURRENT_EPOCH_FEATURES].corr()
fig, ax = plt.subplots(figsize=(6.5,5.2))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(corr)), corr.columns, fontsize=7); ax.grid(False)
fig.colorbar(im, shrink=0.8); plt.title('Aggregate feature correlations'); plt.show()

The lag features are strongly autocorrelated by construction. This is why permutation importance (notebook 07) is read at the level of feature *blocks* rather than trusting individual rankings among correlated columns.